## fit_y0
y0 is the y-coordinate of the incident beam in the laboratory reference frame and needs to be precisely fitted before going further in s3dxrd data processing. This is particularly critical for friedel pairs matching of 2D peaks as the match are found by symmetric scans around y0. If this value is wrong the pairs will not match. 

y0 is fitted using friedel pairs (eta pairs) for the 4D peaks. Once they have been matched, their origin can be relocated in the sample. The relocation depends on y0, and incorrect values yield blurry images of  the sample with artifacts. The game is then to adjust y0 to obtain the sharpest image of the sample.    

In [ ]:
import os, sys, time, glob
start = time.time()

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess


if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.spatial
import ImageD11.sinograms.dataset
from tqdm.autonotebook import tqdm
from scipy.optimize import curve_fit

import ImageD11.friedel_pairs as fp

%matplotlib ipympl

#### Load data

In [ ]:
# dataset path here
dset_path = 'MgO_3_0p1M_12um_0003_dataset.h5'

# initial guess for y0
y0_guess = 11.177

In [ ]:
# read dataset
ds = ImageD11.sinograms.dataset.load(dset_path)
print(ds)

In [ ]:
# get 4D peaks
cf_4d = ds.get_cf_4d()
ds.update_colfile_pars(cf_4d)
print(cf_4d.nrows/1e6, "million peaks read in")

#### match Friedel pairs for 4D peaks

In [ ]:
# Initialize Friedel pairs indexer instance. 
# For more info about how pairing works check this notebook in ImageD11: ImageD11/nbGui/match_friedel_pairs.ipynb
FPI = fp.FriedelPairIndexer(cf_4d, ds,
                            tol_gv = 0.05,
                            tol_eta = 0.2,
                            tol_logI = 0.2,
                            weights = None,
                            n_steps = 25)

In [ ]:
# Match eta pairs. If it takes too long, try lower tol thresholds (tol_gv, tol_eta, tol_logI).
# We don't need to get super high completeness here. With 20% of peaks are paired it should be fine. 

FPI.match_friedel_pairs('eta', drop_unpaired=False, doplot=True, timeout=120)

In [ ]:
# indices of paired peaks
ip, im =fp.get_pairs(cf_4d, 'eta')

Now fit the positions with the y0 guess. Should be faster than finding the pairs.

In [ ]:
# coordinates in the sample. Then use them to build a 2D histogram -> diffraction image of the 4D peaks in the sample 
sx, sy = fp.locate_eta_pairs( cf_4d, (ip,im), ds=ds, y0=y0_guess )
hist_guess = np.histogram2d(sx+y0_guess, sy+y0_guess, bins=ds.ybinedges)[0]

In [ ]:
# plot histogram. This should give an image of the scanned slice
fig, ax = plt.subplots(layout='constrained', figsize=(8,8))
ax.pcolormesh(ds.ybinedges, ds.ybinedges, hist_guess)
ax.set_aspect(1)
ax.set(title=f'y0 guess: {y0_guess}', xlabel='Sample Y axis -->', ylabel='Sample X axis -->')
plt.show()

#### Optimize y0
If y0 is correct, the image above should show sharp bright spots corresponding to the centroids of the grains. If the spots look more like circles or the image is blurry, then y0 is probably wrong

we try this with a range of y0 guesses, and look for a maxima in the standard deviation (sharpest image)

In [ ]:
# guess += 5 from y0_guess, you can change this as needed
y0_guess = 11.22
y0_min = y0_guess - .01
y0_max = y0_guess + .01
n_y0 = 10
y0s = np.linspace(y0_min, y0_max, n_y0)

In [ ]:
# plot reconstructions for different y0s
hists = []

n_cols = 3
n_rows = int(np.ceil(len(y0s) / n_cols))

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else [axes]

for i, y0 in enumerate(tqdm(y0s)):
    sx, sy = fp.locate_eta_pairs(cf_4d, (ip, im), ds=ds, y0=y0)
    hist, _, _ = np.histogram2d(sx + y0, sy + y0, bins=ds.ybinedges)
    hists.append(hist)
    axes[i].pcolormesh(ds.ybinedges, ds.ybinedges, hist, shading='auto')
    axes[i].set_title(f"y0 = {y0:.2f}")
    axes[i].set_xlabel("x bin edges")
    axes[i].set_ylabel("y bin edges")
    axes[i].set_aspect('equal')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()

hists = np.array(hists)

In [ ]:
stdevs = np.std(hists, axis=(1, 2))
maxs = np.max(hists, axis=(1, 2))

In [ ]:
# plot stdev as a function of y0
fig, ax = plt.subplots()
ax.plot(y0s, stdevs, label='stdev')
ax2 = ax.twinx()
ax2.plot(y0s, maxs, color='r', label='max')
fig.legend()
ax.set(xlabel='y0', ylabel='stdev')
ax2.set(ylabel='max')
plt.show()

In [ ]:
# take the results of fit:
y0_final = y0s[np.argmax(stdevs)]
# or manually override from your interpretation of the plot:
# y0_final = 0
print(y0_final)

In [ ]:
# get the final image with fitted y0
sx, sy = fp.locate_eta_pairs( cf_4d, (ip,im), ds, y0 = y0_final )
hist_final =np.histogram2d(sx+y0, sy+y0, bins=ds.ybinedges)[0]

In [ ]:
fig, ax = plt.subplots(layout='constrained', figsize=(8,8))
ax.pcolormesh(ds.ybinedges-y0_final, ds.ybinedges-y0_final, hist_final)
ax.set_aspect(1)
ax.set(title=f'y0 final: {y0_final}', xlabel='Sample Y axis -->', ylabel='Sample X axis -->')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, layout='constrained', figsize=(12,6), sharex=True, sharey=True)
axs[0].pcolormesh(ds.ybinedges, ds.ybinedges, hist_guess)
axs[0].set_aspect(1)
axs[0].set(title=f'y0 guess: {y0_guess}')
axs[1].pcolormesh(ds.ybinedges, ds.ybinedges, hist_final)
axs[1].set_aspect(1)
axs[1].set(title=f'y0 final: {y0_final}')
fig.supxlabel('Sample Y axis -->')
fig.supylabel('Sample X axis -->')
plt.show()